# <font color="#418FDE" size="6.5" uppercase>**TensorFlow Grundlagen**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Erzeugen und untersuchen TensorFlow-Tensoren, Formen und Datentypen. 
- Berechnen Tensoroperationen, Broadcasting und Gradienten mit GradientTape. 
- Erstellen kleine tf.data-Datasets mit Batching, Shuffling und Mapping. 


## **1. Tensoren verstehen**

### **1.1. TensorFlow Version prüfen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_A/image_01_01.jpg?v=1784809486" width="250">



>* TensorFlow-Version vor Tensorarbeit bewusst prüfen
>* Unterschiede zwischen Umgebungen besser einordnen

>* Gleiche Versionen sichern vergleichbare Ergebnisse
>* Versionsangaben helfen bei technischen Problemen

>* Version prüfen schafft technische Orientierung
>* Tensoren dadurch verlässlich interpretieren



### **1.2. Formen und Datentypen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_A/image_01_02.jpg?v=1784809490" width="250">



>* Tensorformen zeigen Achsen und Datenanordnung.
>* Passende Formen verhindern Modell- und Eingabefehler.

>* Datentypen bestimmen Werteart und Speicherdarstellung
>* Sie beeinflussen Genauigkeit, Tempo und Kompatibilität

>* Form und Datentyp gemeinsam prüfen
>* Datenprobleme früh erkennen und vermeiden



In [ ]:
#@title Python-Code - Formen und Datentypen

# Dieses Beispiel untersucht Tensorformen und Datentypen.
# NumPy-Arrays vertreten hier Tensoren ohne TensorFlow-Import.
# Die Ausgabe zeigt Achsen, Formen und Typen.

import numpy as np

# Ein Skalar hat keine Achse.
temperature = np.array(21.5, dtype=np.float32)

# Eine Messreihe hat eine Achse.
sensor_series = np.array([20.1, 20.4, 20.9, 21.5], dtype=np.float32)

# Ein kleines Graustufenbild hat Höhe und Breite.
gray_image = np.array([[0, 80, 160], [40, 120, 255]], dtype=np.uint8)

# Ein Farbbild ergänzt eine Achse für RGB-Kanäle.
rgb_image = np.zeros((2, 3, 3), dtype=np.uint8)
rgb_image[:, :, 0] = gray_image
rgb_image[:, :, 1] = 120
rgb_image[:, :, 2] = 255 - gray_image

# Ein Stapel fügt eine Achse für Beispiele hinzu.
image_batch = np.stack([rgb_image, rgb_image], axis=0)

# Diese Prüfung macht die erwartete Struktur ausdrücklich sichtbar.
expected_shape = (2, 2, 3, 3)
shape_is_expected = image_batch.shape == expected_shape

print("Objekt | Form | Datentyp | Bedeutung")
print(f"Skalar | {temperature.shape} | {temperature.dtype} | ein Wert")
print(f"Reihe | {sensor_series.shape} | {sensor_series.dtype} | Zeitachse")
print(f"Graubild | {gray_image.shape} | {gray_image.dtype} | Höhe x Breite")
print(f"RGB-Bild | {rgb_image.shape} | {rgb_image.dtype} | Höhe x Breite x Kanäle")
print(f"Bildstapel | {image_batch.shape} | {image_batch.dtype} | Beispiele zuerst")
print(f"Erwartete Stapelform erfüllt: {shape_is_expected}")
print("Merke: Form beschreibt die Anordnung, dtype die Wertdarstellung.")



### **1.3. Tensoren nach NumPy**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_A/image_01_03.jpg?v=1784809488" width="250">



>* Tensoren einfach als NumPy-Arrays untersuchen
>* Form und Datentyp bleiben erhalten

>* Tensorwerte mit NumPy einfach prüfen
>* Brücke zwischen TensorFlow und Python-Analyse

>* NumPy-Konvertierung kann Speicher und Zeit kosten
>* Gezielt prüfen, analysieren und Ergebnisse ausgeben



In [ ]:
#@title Python-Code - Tensoren nach NumPy

# Dieses Beispiel zeigt Tensoren als NumPy-Arrays.
# Formen und Datentypen bleiben dabei sichtbar.
# Die Ausgabe vergleicht beide Darstellungen direkt.

import numpy as np

# Wir verwenden NumPy als Tensor-ähnliche Ausgangsdarstellung.
tensor_like = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int32)

# Die Umwandlung erzeugt eine vertraute NumPy-Ansicht der Werte.
numpy_array = np.asarray(tensor_like)

# Diese Prüfung macht die erwartete Form ausdrücklich sichtbar.
if numpy_array.shape != (2, 3):
    raise ValueError("Die Form sollte zwei Zeilen und drei Spalten haben.")

# Kleine Ausgaben zeigen Werte, Form und Datentyp kompakt.
print("Tensor-ähnliche Werte:", tensor_like.tolist())
print("NumPy-Werte:", numpy_array.tolist())
print("Form bleibt gleich:", tensor_like.shape, "->", numpy_array.shape)
print("Datentyp bleibt gleich:", tensor_like.dtype, "->", numpy_array.dtype)
print("NumPy kann direkt weiterrechnen, Spaltensummen:", numpy_array.sum(axis=0).tolist())



## **2. Operationen und Gradienten**

### **2.1. Broadcasting verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_A/image_02_01.jpg?v=1784809480" width="250">



>* Unterschiedliche Tensorformen gemeinsam berechnen
>* Speicher sparen und Code kompakter schreiben

>* Formen von rechts nach links vergleichen
>* Merkmalsvektoren auf Tabellenzeilen anwenden

>* Tensorformen bewusst prüfen, Achsen verstehen
>* Broadcasting vermeidet Fehler und unnötige Schleifen



In [ ]:
#@title Python-Code - Broadcasting verstehen

# Dieses Beispiel zeigt Broadcasting mit kleinen Arrays.
# Formen entscheiden, wie Werte automatisch erweitert werden.
# Die Ausgabe vergleicht erlaubte und unerlaubte Operationen.

import numpy as np

# Eine Tabelle enthält drei Personen und zwei Merkmale.
features = np.array([[10, 100], [20, 200], [30, 300]])

# Ein Korrekturwert pro Spalte passt zu jeder Tabellenzeile.
column_offsets = np.array([1, 10])

# Broadcasting zieht den Vektor von jeder Zeile ab.
corrected = features - column_offsets

# Diese Formprüfung macht die automatische Erweiterung sichtbar.
expected_shape = (3, 2)
if corrected.shape != expected_shape:
    raise ValueError("Die Ergebnisform sollte drei Zeilen und zwei Spalten haben.")

# Ein Spaltenvektor wirkt dagegen einmal pro Zeile.
row_offsets = np.array([[100], [200], [300]])
row_corrected = features - row_offsets

# Nicht jede Formkombination ist für Broadcasting erlaubt.
bad_offsets = np.array([1, 2, 3])
can_broadcast = True
try:
    features - bad_offsets
except ValueError:
    can_broadcast = False

print("Ausgangsform:", features.shape)
print("Spaltenvektor-Form:", column_offsets.shape)
print("Ergebnis nach Spalten-Broadcasting:", corrected.tolist())
print("Zeilenvektor-Form:", row_offsets.shape)
print("Ergebnis nach Zeilen-Broadcasting:", row_corrected.tolist())
print("Form (3,) passt zur Tabelle (3, 2):", can_broadcast)



### **2.2. Gradienten mit GradientTape**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_A/image_02_02.jpg?v=1784809482" width="250">



>* GradientTape zeichnet Rechenschritte automatisch auf
>* Gradienten zeigen sinnvolle Gewichtsänderungen

>* GradientTape verfolgt beobachtete Berechnungen im Aufzeichnungsbereich
>* Empfindlichkeiten ohne manuelle Ableitungen analysieren

>* GradientTape berechnet Gradienten für Trainingsschritte
>* Gradienten steuern Parameter in bessere Richtung



### **2.3. Manueller Gradientenvergleich**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_A/image_02_03.jpg?v=1784809484" width="250">



>* Manuell prüfen, ob GradientTape plausible Ableitungen liefert
>* Gradienten zeigen konkrete Änderungsraten von Tensoroperationen

>* Kleine Beispiele zeigen Gradientenrichtungen anschaulich
>* Vergleiche decken TensorFlow-Fehlerquellen auf

>* Gesamtgradienten entstehen aus verketteten Rechenschritten
>* Vergleiche verbessern Training, Stabilität und Debugging



In [ ]:
#@title Python-Code - Manueller Gradientenvergleich

# Wir vergleichen automatische und manuelle Gradienten.
# Eine einfache Verlustfunktion bleibt gut nachvollziehbar.
# Die Ausgabe zeigt passende Änderungsraten.

import numpy as np

# Diese Werte bilden ein winziges Rechenbeispiel.
x_value = 3.0
weight = 2.0
target = 10.0

# Die Vorhersage und der Fehler entstehen schrittweise.
prediction = weight * x_value
error = prediction - target
loss = error ** 2

# Die Kettenregel liefert den manuellen Gradienten.
manual_gradient = 2.0 * error * x_value

# Eine kleine Änderung nähert den Gradienten numerisch an.
epsilon = 0.0001
loss_plus = ((weight + epsilon) * x_value - target) ** 2
loss_minus = ((weight - epsilon) * x_value - target) ** 2

# Der zentrale Differenzenquotient ist die Vergleichsrechnung.
numeric_gradient = (loss_plus - loss_minus) / (2.0 * epsilon)

# Eine einfache Prüfung schützt vor unerwarteten Formen.
values = np.array([manual_gradient, numeric_gradient], dtype=float)
if values.shape != (2,):
    raise ValueError("Die Gradientenprüfung erwartet genau zwei Werte.")

# Die Ausgabe vergleicht Verlust und beide Gradienten.
print("Manueller Gradientenvergleich für loss = (weight * x - target)^2")
print(f"x={x_value:.1f}, weight={weight:.1f}, target={target:.1f}")
print(f"Vorhersage={prediction:.1f}, Fehler={error:.1f}, Verlust={loss:.1f}")
print(f"Manueller Gradient: {manual_gradient:.4f}")
print(f"Numerischer Gradient: {numeric_gradient:.4f}")
print(f"Abweichung: {abs(manual_gradient - numeric_gradient):.8f}")



## **3. tf data Grundlagen**

### **3.1. Dataset erstellen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_A/image_03_01.jpg?v=1784809491" width="250">



>* Datasets strukturieren Daten für Modelle
>* Sie liefern Beispiele wie ein Förderband

>* Daten TensorFlow-gerecht aus verschiedenen Quellen vorbereiten
>* Als Datenstrom für weitere Verarbeitung nutzen

>* Klare Dataset-Struktur verhindert Trainingsfehler
>* Datenverständnis formt lernbare Beispiele



In [ ]:
#@title Python-Code - Dataset erstellen

# Dieses Beispiel erstellt ein kleines Dataset.
# Es zeigt Mischen, Batching und Mapping.
# Die Ausgabe macht jeden Schritt sichtbar.

import numpy as np

# Kleine Arrays stehen hier für Eingaben und Zielwerte.
features = np.array([10, 20, 30, 40, 50, 60], dtype=np.float32)
labels = np.array([1, 2, 3, 4, 5, 6], dtype=np.float32)

# Jedes Dataset-Element ist ein zusammengehöriges Beispielpaar.
dataset = list(zip(features, labels))
print("Original: 6 einzelne Beispielpaare")

# Eine feste Zufallsquelle macht das Mischen reproduzierbar.
rng = np.random.default_rng(42)
shuffled_indices = rng.permutation(len(dataset))
shuffled_dataset = [dataset[index] for index in shuffled_indices]

# Mapping verändert jedes Beispiel auf dieselbe Weise.
mapped_dataset = []
for feature, label in shuffled_dataset:
    mapped_dataset.append((feature / 10.0, label))

# Batching gruppiert mehrere Beispiele für einen Trainingsschritt.
batch_size = 2
batches = []
for start in range(0, len(mapped_dataset), batch_size):
    batches.append(mapped_dataset[start:start + batch_size])

# Eine einfache Prüfung verhindert unbemerkte Strukturfehler.
expected_batches = 3
if len(batches) != expected_batches:
    raise ValueError("Die Anzahl der Batches stimmt nicht.")

# Die Ausgabe zeigt die Pipeline in kompakter Form.
print("Nach Shuffle, Map und Batch:")
for batch_number, batch in enumerate(batches, start=1):
    batch_features = [round(float(item[0]), 1) for item in batch]
    batch_labels = [int(item[1]) for item in batch]
    print(f"Batch {batch_number}: x={batch_features}, y={batch_labels}")

# Die letzte Zeile bestätigt die typische Dataset-Struktur.
print("Fertig: jedes Batch enthält Eingaben und passende Zielwerte.")



### **3.2. Batching und Shuffling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_A/image_03_02.jpg?v=1784809493" width="250">



>* Batching gruppiert Beispiele für gemeinsame Verarbeitung
>* Effizientere Hardware, stabileres Lernen

>* Shuffling mischt Beispiele vor dem Training
>* Batches werden vielfältiger und weniger verzerrt

>* Erst mischen, dann repräsentative Batches bilden
>* Puffer- und Batchgröße steuern Speicherbedarf



In [ ]:
#@title Python-Code - Batching und Shuffling

# Dieses Beispiel zeigt Batching und Shuffling.
# Kleine Zahlen ersetzen ein echtes Trainingsdataset.
# Die Ausgabe vergleicht geordnete und gemischte Batches.

import numpy as np

# Wir erzeugen zwölf einfache Beispiele mit passenden Labels.
features = np.arange(12)
labels = features % 3

# Diese Prüfung schützt vor unpassenden Beispielzahlen.
if len(features) != len(labels):
    raise ValueError("Features und Labels müssen gleich lang sein.")

# Ohne Shuffling bleiben benachbarte Beispiele zusammen.
batch_size = 4
ordered_indices = np.arange(len(features))

# Mit festem Seed ist die zufällige Reihenfolge reproduzierbar.
rng = np.random.default_rng(42)
shuffled_indices = rng.permutation(len(features))

# Diese Funktion baut kleine Batches aus einer Indexreihenfolge.
def make_batches(indices, values, targets, size):
    batches = []
    for start in range(0, len(indices), size):
        batch_indices = indices[start:start + size]
        batches.append((values[batch_indices], targets[batch_indices]))
    return batches

ordered_batches = make_batches(ordered_indices, features, labels, batch_size)
shuffled_batches = make_batches(shuffled_indices, features, labels, batch_size)

# Die Ausgabe zeigt, welche Beispiele in jedem Batch landen.
print("Geordnete Batches: Beispiele -> Labels")
for number, batch in enumerate(ordered_batches, start=1):
    print(f"Batch {number}: {batch[0].tolist()} -> {batch[1].tolist()}")

print("Gemischte Batches: Beispiele -> Labels")
for number, batch in enumerate(shuffled_batches, start=1):
    print(f"Batch {number}: {batch[0].tolist()} -> {batch[1].tolist()}")

# Diese letzte Zeile bestätigt die zentrale Beobachtung.
print("Shuffling ändert die Reihenfolge, Batching gruppiert danach.")



### **3.3. Mapping praktisch üben**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_A/image_03_03.jpg?v=1784809495" width="250">



>* Mapping bereitet jedes Dataset-Element systematisch vor
>* Transformationen bleiben direkt in der Pipeline

>* Mapping macht Transformationen einheitlich und nachvollziehbar
>* Reihenfolge und Batch-Ebene bewusst wählen

>* Mit kleinen Datasets Mapping nachvollziehbar üben
>* Mapping als Kern sauberer Datenpipelines verstehen



In [ ]:
#@title Python-Code - Mapping praktisch üben

# Dieses Beispiel übt Mapping in einer kleinen Datenpipeline.
# Jede Transformation wird sichtbar und leicht nachvollziehbar.
# Am Ende entstehen gemischte und gebündelte Trainingsbeispiele.

import numpy as np

# Kleine Rohdaten ersetzen hier ein einfaches tf.data-Dataset.
temperatures_celsius = np.array([18.0, 21.5, 19.0, 24.0, 22.5, 20.0])
labels = np.array([0, 1, 0, 1, 1, 0])

# Diese Prüfung macht die Pipeline für Anfänger robuster.
if len(temperatures_celsius) != len(labels):
    raise ValueError("Temperaturen und Labels müssen gleich lang sein.")

# Mapping wendet dieselbe Umrechnung auf jedes Element an.
temperatures_fahrenheit = temperatures_celsius * 9.0 / 5.0 + 32.0
scaled_temperatures = (temperatures_fahrenheit - 32.0) / 50.0

# Shuffling verändert die Reihenfolge, aber nicht die Zuordnung.
rng = np.random.default_rng(42)
shuffled_indices = rng.permutation(len(scaled_temperatures))

mapped_features = scaled_temperatures[shuffled_indices]
mapped_labels = labels[shuffled_indices]

# Batching fasst vorbereitete Beispiele zu kleinen Gruppen zusammen.
batch_size = 2
feature_batches = mapped_features.reshape(3, batch_size)
label_batches = mapped_labels.reshape(3, batch_size)

print("Rohwerte in °C:", temperatures_celsius.tolist())
print("Nach Mapping skaliert:", np.round(scaled_temperatures, 2).tolist())
print("Gemischte Reihenfolge:", shuffled_indices.tolist())
print("Batch 1 Merkmale:", np.round(feature_batches[0], 2).tolist())
print("Batch 1 Labels:", label_batches[0].tolist())
print("Ergebnis: Mapping bereitet jedes Beispiel vor dem Batching gleich vor.")



# <font color="#418FDE" size="6.5" uppercase>**TensorFlow Grundlagen**</font>


In this lecture, you learned to:
- Erzeugen und untersuchen TensorFlow-Tensoren, Formen und Datentypen. 
- Berechnen Tensoroperationen, Broadcasting und Gradienten mit GradientTape. 
- Erstellen kleine tf.data-Datasets mit Batching, Shuffling und Mapping. 

In the next Lecture (Lecture B), we will go over 'Dichte Keras-Modelle'